# 03 · Outlier score development

How the peer-adjusted score is built and what it responds to. This notebook
never looks at the exclusion outcome; evaluation is in notebook 04.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from _setup import OCFG, WH, banner
from partd_risk.modeling.outlier_score import score_prescribers
from partd_risk.modeling.peer_adjust import transform_metric

banner()

In [ ]:
features = WH.read("marts", "mart_outlier_features")
outcome_cols = [c for c in features.columns if "exclu" in c]
X = features.drop(columns=outcome_cols)
print(f"{len(X):,} cohort prescribers, {X['peer_group_id'].nunique():,} peer groups")
X["peer_group_level"].value_counts(normalize=True)

## Feature distributions after transformation

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
for ax, spec in zip(axes.flat, OCFG.features, strict=False):
    transform_metric(X[spec.column], spec.transform, OCFG.logit_eps).dropna().plot.hist(
        bins=60, ax=ax
    )
    ax.set_title(f"{spec.name} ({spec.transform})", fontsize=9)
fig.tight_layout()

## Score

In [ ]:
result = score_prescribers(X, OCFG)
diag = result.adjustment.diagnostics()
diag[["feature", "transform", "n", "coverage", "r2_within", "residual_scale"]]

`r2_within` is the share of within-peer-group variance explained by patient
case-mix. The case-mix coefficients below show the direction of each
adjustment (e.g. sicker panels -> higher cost per fill).

In [ ]:
diag.set_index("feature").filter(like="beta__").T

## How correlated are the peer-adjusted z-scores?

In [ ]:
z = result.adjustment.z
z.corr().round(2)

## Score distribution and top-1% composition

In [ ]:
scores = result.scores
ax = scores["outlier_score"].plot.hist(bins=100, log=True)
ax.axvline(scores.loc[scores["is_top_1pct"], "outlier_score"].min(), color="black")
ax.set_title("Outlier score (log count); line = top-1% threshold")

In [ ]:
top = scores[scores["is_top_1pct"]]
pd.concat(
    [
        top["top_feature"].value_counts(normalize=True).rename("share_of_top_1pct"),
        top.groupby("top_feature")["top_feature_z"].median().rename("median_z"),
    ],
    axis=1,
)

## Peer adjustment vs. none: who moves?

In [ ]:
ranks = pd.DataFrame(
    {
        "adjusted_rank": scores["outlier_score"].rank(ascending=False),
        "unadjusted_rank": scores["unadjusted_score"].rank(ascending=False),
        "specialty": X["prescriber_specialty"],
    }
)
cut = int(np.ceil(0.01 * len(ranks)))
only_unadj = ranks[(ranks["unadjusted_rank"] <= cut) & (ranks["adjusted_rank"] > cut)]
only_adj = ranks[(ranks["adjusted_rank"] <= cut) & (ranks["unadjusted_rank"] > cut)]
pd.concat(
    [
        only_unadj["specialty"].value_counts().rename("leave_top1_after_adjustment"),
        only_adj["specialty"].value_counts().rename("enter_top1_after_adjustment"),
    ],
    axis=1,
).fillna(0).astype(int).head(15)